In [2]:
import pandas as pd
import numpy as np
from datasets import load_dataset
import tiktoken
import json
from collections import Counter
import matplotlib.pyplot as plt
import seaborn as sns
import re
from groq import Groq
import os
from dotenv import load_dotenv
import random

load_dotenv()  # Load environment variables from a .env file if present
client = Groq(api_key=os.getenv("groq_api_key"))

d:\Work\Github\google-tunix-kaggle\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import os
import time
import pandas as pd
from groq import Groq
from tqdm import tqdm
from datetime import datetime

class GroqBatchProcessor:
    """Process batches of  problems with Groq OSS-120B"""
    
    def __init__(
        self, 
        target_model="openai/gpt-oss-120b",
        system_prompt="",
        max_retries=3,
        retry_delay=2
    ):
        self.client = Groq(api_key=os.getenv("GROQ_API_KEY"))
        self.target_model = target_model
        self.system_prompt = system_prompt
        self.max_retries = max_retries
        self.retry_delay = retry_delay
        
        print(f"System prompt set to: {self.system_prompt}")
        
    def process_single(self, problem_text, reasoning_effort="medium", max_tokens=2048):
        """Process a single problem with retries"""
        
        messages = [
            {"role": "system", "content": self.system_prompt},
            {"role": "user", "content": f"{problem_text}"}
        ]
        
        for attempt in range(self.max_retries):
            try:
                chat_completion = self.client.chat.completions.create(
                    model=self.target_model,
                    messages=messages,
                    temperature=0.9,
                    max_completion_tokens=max_tokens,
                    top_p=1,
                    reasoning_effort=reasoning_effort,
                    stream=False,
                    stop=None
                )
                
                msg = chat_completion.choices[0].message
                
                # Extract reasoning and response
                response = msg.content.strip() if msg.content else ""
                reasoning = msg.reasoning.strip() if hasattr(msg, 'reasoning') and msg.reasoning else ""
                
                # Get token counts
                usage = chat_completion.usage
                input_tokens = usage.prompt_tokens if hasattr(usage, 'prompt_tokens') else 0
                output_tokens = usage.completion_tokens if hasattr(usage, 'completion_tokens') else 0
                
                return {
                    'reasoning': reasoning,  # OSS-120B reasoning trace
                    'response': response,    # Final solution/answer
                    'input_tokens': input_tokens,
                    'output_tokens': output_tokens,
                    'total_tokens': input_tokens + output_tokens,
                    'success': True,
                    'error': None
                }
                
            except Exception as e:
                if attempt < self.max_retries - 1:
                    print(f"Attempt {attempt + 1} failed: {e}. Retrying in {self.retry_delay}s...")
                    time.sleep(self.retry_delay)
                else:
                    print(f"Failed after {self.max_retries} attempts: {e}")
                    return {
                        'reasoning': None,
                        'response': None,
                        'input_tokens': 0,
                        'output_tokens': 0,
                        'total_tokens': 0,
                        'success': False,
                        'error': str(e)
                    }
        
    def process_batch(
        self, 
        df, 
        input_column='input',
        uid_column='uid',
        reasoning_effort="medium",
        max_tokens=2048,
        batch_size=10,
        save_interval=100,
        output_file='oss120b_results.parquet'
    ):
        """Process batch and save reasoning + response"""
        
        results = []
        result_master=[]
        start_time = time.time()
        
        print(f"Processing {len(df)} problems with {self.target_model}")
        print(f"Reasoning effort: {reasoning_effort}, Max tokens: {max_tokens}\n")
        

        
        for idx, row in tqdm(df.iterrows(), total=len(df), desc="Processing"):
            problem = row[input_column]
            
            # Process single problem
            result = self.process_single(
                problem_text=problem,
                reasoning_effort=reasoning_effort,
                max_tokens=max_tokens
            )
            
            # Add original data + metadata
            result.update({
                'uid': row[uid_column],
                'input': problem,  # Original problem
            })
            
            results.append(result)
            
            
            # Rate limiting
            if (idx + 1) % batch_size == 0:
                #Add batch to master
                result_master.extend(results)
                #Clear results for next batch
                results = []
                
                time.sleep(1)
            
            # Save checkpoint
            if (idx + 1) % save_interval == 0:
                self._save_checkpoint(result_master, output_file, idx + 1)
        
        
        # Add any leftover batch results
        if len(results) > 0:
            result_master.extend(results)

        # Final save with ALL columns
        df_results = pd.DataFrame(result_master)
        df_results.to_parquet(output_file, index=False)
        
        # Print summary
        elapsed = time.time() - start_time
        self._print_summary(df_results, elapsed)
        
        return df_results
    
    def _save_checkpoint(self, results, output_file, count):
        """Save intermediate results"""
        checkpoint_file = output_file.replace('.parquet', f'_checkpoint_{count}.parquet')
        df_temp = pd.DataFrame(results)
        df_temp.to_parquet(checkpoint_file, index=False)
        print(f"\n✓ Checkpoint saved: {checkpoint_file} ({len(results)} records)")
    
    def _print_summary(self, df_results, elapsed_time):
        """Print processing summary"""
        total = len(df_results)
        successful = df_results['success'].sum()
        failed = total - successful
        
        # Check reasoning availability
        has_reasoning = df_results[df_results['success'] == True]['reasoning'].notna().sum()
        
        avg_input_tokens = df_results[df_results['success'] == True]['input_tokens'].mean()
        avg_output_tokens = df_results[df_results['success'] == True]['output_tokens'].mean()
        total_tokens = df_results['total_tokens'].sum()
        
        print("\n" + "="*60)
        print("PROCESSING SUMMARY")
        print("="*60)
        print(f"Total processed: {total}")
        print(f"Successful: {successful} ({successful/total*100:.1f}%)")
        print(f"Failed: {failed} ({failed/total*100:.1f}%)")
        print(f"Has reasoning: {has_reasoning}")
        print(f"\nToken Usage:")
        print(f"  Avg input tokens: {avg_input_tokens:.0f}")
        print(f"  Avg output tokens: {avg_output_tokens:.0f}")
        print(f"  Total tokens used: {total_tokens:,}")
        print(f"\nTime: {elapsed_time/60:.1f} minutes")
        print(f"Rate: {total/(elapsed_time/60):.1f} problems/min")
        print("="*60)

In [7]:
import re

def extract_answer(text):
    """Extract answer from multiple common formats"""
    if text is None or not isinstance(text, str):
        return None
    
    # Pattern 1: \boxed{answer}
    match = re.search(r'\\boxed\{([^}]+)\}', text)
    if match:
        return match.group(1).strip()
    
    # Pattern 2: **Answer:** followed by content
    match = re.search(r'\*\*Answer:\*\*\s*(.+?)(?:\n|\.|$)', text, re.IGNORECASE)
    if match:
        return match.group(1).strip()
    
    # Pattern 3: "Answer: X" or "The answer is X"
    match = re.search(r'(?:answer is|answer:)\s*\*?\*?(.+?)(?:\.|$)', text, re.IGNORECASE)
    if match:
        return match.group(1).strip()
    
    # Pattern 4: Last sentence with numerical result (for math)
    # Look for patterns like "Albert eats 48 pieces" or "Total = $5,250"
    sentences = text.split('.')
    for sentence in reversed(sentences):
        # Find numbers with units or currency
        match = re.search(r'(\$?[\d,]+(?:\.\d+)?(?:\s*(?:pieces|slices|dollars|items|units|people))?)', sentence)
        if match:
            return match.group(1).strip()
    
    return None


def latex_to_plain(text: str) -> str:
    if not isinstance(text, str):
        return text

    # Remove math delimiters
    text = re.sub(r'\\\(|\\\)|\\\[|\\\]', '', text)

    # Replace LaTeX thousands separator {,} -> ,
    text = text.replace("{,}", ",")

    # Replace \text{...} with the content
    text = re.sub(r'\\text\{([^}]*)\}', r'\1', text)

    # Remove any remaining { } without destroying content
    text = text.replace("{", "").replace("}", "")

    # Remove common LaTeX commands
    text = re.sub(r'\\(frac|cdot|times|left|right|big|begin|end|quad|qquad)[^ ]*', '', text)

    # Replace LaTeX fractions \frac{a}{b} → a/b
    text = re.sub(r'\\frac\s*([^}]*)\s*/\s*([^}]*)', r'\1/\2', text)

    # Replace ^ and _ braces: x^{2} → x^2
    text = re.sub(r'\^\{([^}]*)\}', r'^\1', text)
    text = re.sub(r'_\{([^}]*)\}', r'_\1', text)

    # Collapse multiple spaces
    text = re.sub(r'\s+', ' ', text).strip()

    return text


### 1. Getting the distillation reasoning and response from OSS120B model for MATH

    - reasoning = medium
    - max tokens = 1900 (output)
    - temperature = 0.9

- We will generate reasoning and response for all questions.
- We will not add any complex prompts to it. Prompt is simple : "You are an expert in math. Provide clear, concise solutions to these math problems within the token limit."
- Some answers might be wrong, some answers might be truncated. We flag these at the end

In [23]:
COMPETITIVE_SYSTEM_PROMPT = """You are a competitive programming expert. Solve this problem CONCISELY.

REQUIREMENTS:
- Write clean, efficient  code
- Code must be inside ```code ``` block

Try to keep response focused and under 600 words."""

MBPP_SYSTEM_PROMPT = """You are an expert Python programmer. Write a function that solves the given problem.

REQUIREMENTS:
- Explain your approach
- Write a clean Python function with correct signature
- Code must be inside ```python ``` block
- Ensure the solution handles all edge cases

Try to Keep response focused and under 600 words."""

CODE_alpaca_prompt = """You are an expert programmer. Follow the instruction to write clean, working code.

REQUIREMENTS:
- Explain your solution approach
- Write clean, well-structured  code
- Code must be inside ```code ``` block

Try to Keep response focused and under 500 words."""

python_alpaca_prompt = """You are an expert Python programmer. Follow the instruction to write clean, working code.

REQUIREMENTS:
- Explain your solution approach
- Write clean, well-structured Python code
- Code must be inside ```python ``` block

Try to Keep response focused and under 500 words."""

In [27]:
##Load math data
code_data = pd.read_parquet('../data/raw-data/code_base_dataset.parquet')
code_data.groupby('source').size()

source
codeparrot/apps                                  5000
google-research-datasets/mbpp                     974
iamtarun/python_code_instructions_18k_alpaca    18612
nvidia/OpenCodeReasoning                        10000
sahil2801/CodeAlpaca-20k                        20022
dtype: int64

In [28]:
mbpp=code_data[code_data['source']=='google-research-datasets/mbpp']
code_alpaca = code_data[code_data['source']=='sahil2801/CodeAlpaca-20k']
python_code_alpaca = code_data[code_data['source']=='iamtarun/python_code_instructions_18k_alpaca']
codeparrot_apps = code_data[code_data['source']=='codeparrot/apps']
opencode_reasoning = code_data[code_data['source']=='nvidia/OpenCodeReasoning']

print(f"MBPP data: {len(mbpp)} records")
print(f"Code Alpaca data: {len(code_alpaca)} records")
print(f"Python Code Alpaca data: {len(python_code_alpaca)} records")
print(f"CodeParrot Apps data: {len(codeparrot_apps)} records")
print(f"OpenCode Reasoning data: {len(opencode_reasoning)} records")

MBPP data: 974 records
Code Alpaca data: 20022 records
Python Code Alpaca data: 18612 records
CodeParrot Apps data: 5000 records
OpenCode Reasoning data: 10000 records


In [25]:
MBPP_SYSTEM_PROMPT

'You are an expert Python programmer. Write a function that solves the given problem.\n\nREQUIREMENTS:\n- Explain your approach\n- Write a clean Python function with correct signature\n- Code must be inside ```python ``` block\n- Ensure the solution handles all edge cases\n\nTry to Keep response focused and under 600 words.'

In [33]:
# Initialize
processor = GroqBatchProcessor(
    target_model="openai/gpt-oss-120b",
    system_prompt=MBPP_SYSTEM_PROMPT,
    
)

# Process your filtered dataset
mbpp_results = processor.process_batch(
    df=mbpp.reset_index(drop=True),
    input_column='input',
    uid_column='uid',
    reasoning_effort='medium',
    max_tokens=1930,
    batch_size=100,
    save_interval=500,  # Save every 500 records
    output_file='../data/code-distillation/checkpoints/mathcode_distillation_with_reasoning_mbpp.parquet'
)
    
# Verify columns
print("\nColumns in results:")
print(mbpp_results.columns.tolist())

System prompt set to: You are an expert Python programmer. Write a function that solves the given problem.

REQUIREMENTS:
- Explain your approach
- Write a clean Python function with correct signature
- Code must be inside ```python ``` block
- Ensure the solution handles all edge cases

Try to Keep response focused and under 600 words.
Processing 974 problems with openai/gpt-oss-120b
Reasoning effort: medium, Max tokens: 1930



Processing:  51%|█████▏    | 500/974 [23:02<25:58,  3.29s/it]


✓ Checkpoint saved: ../data/code-distillation/checkpoints/mathcode_distillation_with_reasoning_mbpp_checkpoint_500.parquet (500 records)


Processing: 100%|██████████| 974/974 [44:40<00:00,  2.75s/it]


PROCESSING SUMMARY
Total processed: 974
Successful: 974 (100.0%)
Failed: 0 (0.0%)
Has reasoning: 974

Token Usage:
  Avg input tokens: 155
  Avg output tokens: 1175
  Total tokens used: 1,295,359

Time: 44.7 minutes
Rate: 21.8 problems/min

Columns in results:
['reasoning', 'response', 'input_tokens', 'output_tokens', 'total_tokens', 'success', 'error', 'uid', 'input']


In [34]:
competetive_df = pd.concat([codeparrot_apps.reset_index(drop=True), opencode_reasoning.reset_index(drop=True)], ignore_index=True)

In [36]:
# Initialize
processor = GroqBatchProcessor(
    target_model="openai/gpt-oss-120b",
    system_prompt=COMPETITIVE_SYSTEM_PROMPT
    
)

# Process your filtered dataset
competetive_df_results = processor.process_batch(
    df=competetive_df.reset_index(drop=True).sample(1000, random_state=42),
    input_column='input',
    uid_column='uid',
    reasoning_effort='medium',
    max_tokens=1930,
    batch_size=100,
    save_interval=500,  # Save every 1000 records
    output_file='../data/code-distillation/checkpoints/code_distillation_with_reasoning_competitive.parquet'
)

System prompt set to: You are a competitive programming expert. Solve this problem CONCISELY.

REQUIREMENTS:
- Write clean, efficient  code
- Code must be inside ```code ``` block

Try to keep response focused and under 600 words.
Processing 1000 problems with openai/gpt-oss-120b
Reasoning effort: medium, Max tokens: 1930



Processing:   0%|          | 1/1000 [00:05<1:25:08,  5.11s/it]


✓ Checkpoint saved: ../data/code-distillation/checkpoints/code_distillation_with_reasoning_competitive_checkpoint_11500.parquet (1 records)


Processing:  10%|█         | 105/1000 [06:54<1:05:43,  4.41s/it]


✓ Checkpoint saved: ../data/code-distillation/checkpoints/code_distillation_with_reasoning_competitive_checkpoint_10000.parquet (105 records)


Processing:  85%|████████▍ | 846/1000 [54:44<09:18,  3.63s/it]  


✓ Checkpoint saved: ../data/code-distillation/checkpoints/code_distillation_with_reasoning_competitive_checkpoint_11000.parquet (846 records)


Processing: 100%|██████████| 1000/1000 [1:04:30<00:00,  3.87s/it]


PROCESSING SUMMARY
Total processed: 1000
Successful: 1000 (100.0%)
Failed: 0 (0.0%)
Has reasoning: 1000

Token Usage:
  Avg input tokens: 546
  Avg output tokens: 1688
  Total tokens used: 2,233,797

Time: 64.5 minutes
Rate: 15.5 problems/min


In [37]:
# Initialize
processor = GroqBatchProcessor(
    target_model="openai/gpt-oss-120b",
    system_prompt=CODE_alpaca_prompt
    
)

# Process your filtered dataset
code_alpaca_results = processor.process_batch(
    df=code_alpaca.reset_index(drop=True).sample(3000, random_state=42),
    input_column='input',
    uid_column='uid',
    reasoning_effort='medium',
    max_tokens=1930,
    batch_size=100,
    save_interval=1000,  # Save every 1000 records
    output_file='../data/code-distillation/checkpoints/code_distillation_with_reasoning_code_alpaca.parquet'
)

System prompt set to: You are an expert programmer. Follow the instruction to write clean, working code.

REQUIREMENTS:
- Explain your solution approach
- Write clean, well-structured  code
- Code must be inside ```code ``` block

Try to Keep response focused and under 500 words.
Processing 3000 problems with openai/gpt-oss-120b
Reasoning effort: medium, Max tokens: 1930



Processing: 100%|██████████| 3000/3000 [1:13:29<00:00,  1.47s/it]


PROCESSING SUMMARY
Total processed: 3000
Successful: 3000 (100.0%)
Failed: 0 (0.0%)
Has reasoning: 3000

Token Usage:
  Avg input tokens: 156
  Avg output tokens: 644
  Total tokens used: 2,398,791

Time: 73.5 minutes
Rate: 40.8 problems/min


In [38]:
# Initialize
processor = GroqBatchProcessor(
    target_model="openai/gpt-oss-120b",
    system_prompt=python_alpaca_prompt
    
)

# Process your filtered dataset
python_alpaca_results = processor.process_batch(
    df=python_code_alpaca.reset_index(drop=True).sample(3000, random_state=42),
    input_column='input',
    uid_column='uid',
    reasoning_effort='medium',
    max_tokens=1930,
    batch_size=100,
    save_interval=1000,  # Save every 1000 records
    output_file='../data/code-distillation/checkpoints/code_distillation_with_reasoning_python_code_alpaca.parquet'
)

System prompt set to: You are an expert Python programmer. Follow the instruction to write clean, working code.

REQUIREMENTS:
- Explain your solution approach
- Write clean, well-structured Python code
- Code must be inside ```python ``` block

Try to Keep response focused and under 500 words.
Processing 3000 problems with openai/gpt-oss-120b
Reasoning effort: medium, Max tokens: 1930



Processing:   9%|▉         | 273/3000 [06:36<1:10:19,  1.55s/it]


✓ Checkpoint saved: ../data/code-distillation/checkpoints/code_distillation_with_reasoning_python_code_alpaca_checkpoint_6000.parquet (273 records)


Processing: 100%|██████████| 3000/3000 [1:17:07<00:00,  1.54s/it]


PROCESSING SUMMARY
Total processed: 3000
Successful: 3000 (100.0%)
Failed: 0 (0.0%)
Has reasoning: 3000

Token Usage:
  Avg input tokens: 308
  Avg output tokens: 660
  Total tokens used: 2,904,850

Time: 77.1 minutes
Rate: 38.9 problems/min


In [39]:
code_final = pd.concat([code_alpaca_results,python_alpaca_results,competetive_df_results,mbpp_results],ignore_index=True)

In [40]:
code_final1 = pd.merge(code_data,code_final.drop(columns=['input']), on='uid', how='inner')

In [42]:
##Save the math base dataset
code_final1.to_parquet('../data/code-distillation/code_distillation_dataset.parquet')
code_final1.to_csv('../data/code-distillation/code_distillation_dataset.csv', index=False)

In [44]:
code_final1.sample(30)

,input,source_answer,ground_truth,split,source,domain,problem_type,question_type,difficulty,uid,reasoning,response,input_tokens,output_tokens,total_tokens,success,error
2260,Convert the following JSON data to an HTML tab...,<table>\n <tr>\n <th>Fruit</th>\n <th>V...,,train,sahil2801/CodeAlpaca-20k,code,code generation,code generation by instruction,UNKNOWN_DIFFICULTY,code18164,We need to produce HTML table from JSON. Provi...,**Solution Overview**\n\n1. **Parse the JSON**...,173,846,1019,True,None
6026,Below is an instruction that describes a task....,def is_even(number): \n if(number % 2 == 0)...,,train,iamtarun/python_code_instructions_18k_alpaca,code,python code generation,python code generation by instruction,UNKNOWN_DIFFICULTY,code42524,The user wants a response that appropriately c...,**Solution Approach**\n\nTo decide if an integ...,205,520,725,True,None
859,Write a function to find the smallest integers...,import heapq as hq\r\ndef heap_queue_smallest(...,"[""assert heap_queue_smallest( [25, 35, 22, 85,...",test,google-research-datasets/mbpp,code,Algorithmic reasoning,Short Python algorithm tasks,UNKNOWN_DIFFICULTY,code860,"The user asks: ""Write a function to find the s...",**Solution Explanation**\n\nWe need to return ...,158,912,1070,True,None
6274,Below is an instruction that describes a task....,# Getting date from user\ndate_string = input(...,,train,iamtarun/python_code_instructions_18k_alpaca,code,python code generation,python code generation by instruction,UNKNOWN_DIFFICULTY,code43985,The user wants a response that appropriately c...,**Solution Approach**\n\n1. **Read the date** ...,286,830,1116,True,None
209,Write a function to iterate over elements repe...,from collections import Counter\r\ndef count_v...,"[""assert count_variable(4,2,0,-2)==['p', 'p', ...",train,google-research-datasets/mbpp,code,Algorithmic reasoning,Short Python algorithm tasks,UNKNOWN_DIFFICULTY,code210,"We need to understand the problem: ""Write a fu...",**Solution Explanation**\n\nFor a given iterab...,155,1166,1321,True,None
1314,A tree is an undirected graph with exactly one...,import sys\nfrom sys import stdin\nfrom collec...,,train,nvidia/OpenCodeReasoning,code,code_contests,codeforces,VERY_HARD,code6479,We need for each k max number of vertex-disjoi...,,427,1930,2357,True,None
2315,Construct a T-SQL query to calculate the avera...,SELECT AVG(price) FROM products;,,train,sahil2801/CodeAlpaca-20k,code,code generation,code generation by instruction,UNKNOWN_DIFFICULTY,code18491,We need to respond with explanation and T-SQL ...,**Solution Approach**\n\n1. **Understand the r...,200,383,583,True,None
7190,Below is an instruction that describes a task....,def odd_numbers(numbers):\n result = 0 \n ...,,train,iamtarun/python_code_instructions_18k_alpaca,code,python code generation,python code generation by instruction,UNKNOWN_DIFFICULTY,code49668,The user wants a response that appropriately c...,**Solution Approach**\n\n1. **Iterate through ...,239,409,648,True,None
6190,Below is an instruction that describes a task....,"for i in range(5):\n print(""I love coding!"")",,train,iamtarun/python_code_instructions_18k_alpaca,code,python code generation,python code generation by instruction,UNKNOWN_DIFFICULTY,code43555,"The user asks: ""Below is an instruction that d...",**Solution Approach**\n\nWe need to repeat a s...,191,195,386,True,None
5336,Below is an instruction that describes a task....,import xml.etree.ElementTree as ET\n\ndef extr...,,train,iamtarun/python_code_instructions_18k_alpaca,code,python code generation,python code generation by instruction,UNKNOWN_DIFFICULTY,code38357,"The user asks: ""Below is an instruction that d...",**Solution Approach**\n\n1. **Parsing the XML*...,249,869,1118,True,None


In [47]:
chk = pd.read_parquet('../data/code-distillation/code_distillation_dataset.parquet')
chk.shape

(7974, 17)